In [1]:
import numpy as np
import pandas as pd
from scipy import signal
from scipy.io import wavfile
from scipy.fft import fftshift
import matplotlib.pyplot as plt
import librosa
from scipy.signal import hilbert, butter, lfilter
import os
import textgrid
import sys
import shutil
import math

We have around:
* *5h32min* of negative audio samples
* *1h* of positive audio samples

### Positives

In [222]:
# Divide recordings into 500ms clips 
folder = "/Users/yasminebenhamadi/YellowHammer/Training_set_YH_songs/ALL_Songs_CUT/"
positive_folder = "/Users/yasminebenhamadi/YellowHammer/Training_set_YH_songs/positive/"

if not os.path.exists(positive_folder):
    os.makedirs(positive_folder)

    files = os.listdir(folder)
    files = [file for file in files if ".wav" in file]
    files = [file for file in files if "speaker" not in file]

    jj = []

    for sound_file in files:
        samplerate, data = wavfile.read(folder+sound_file)

        if data.shape[0]/samplerate >= 0.5:
            nb_clips = int(data.shape[0]/(samplerate*0.5))
            nb_clips = np.min([nb_clips,3])
            jj.append(nb_clips)

            start = 0
            end = int(0.5*samplerate)
            for clip in range(nb_clips):
                data_clipped = data[start:end]
                wavfile.write(positive_folder+"clip_"+str(clip+1)+"_"+sound_file, samplerate, data_clipped)
                start=end
                end = end + int(0.5*samplerate)
else:
    print("Folder: "+positive_folder+" exists already.")

Folder: /Users/yasminebenhamadi/YellowHammer/Training_set_YH_songs/positive/ exists already.


### Negatives

In [223]:
# Extract negatives from the full recordings

grid_folder = "/Users/yasminebenhamadi/YellowHammer/Training_set_YH_songs/recordings_textgrid_on_axis/"
negative_folder = "/Users/yasminebenhamadi/YellowHammer/Training_set_YH_songs/negative/"

if not os.path.exists(negative_folder):
    os.makedirs(negative_folder)

    files = os.listdir(grid_folder)
    files = [f for f in files if ".wav" in f]
    files = [f for f in files if "speaker" not in f]
    
    for sound_file in files:
        samplerate, data = wavfile.read(grid_folder+sound_file)
        
        grid_file = grid_folder + sound_file[:-4]+".TextGrid"
        tg = textgrid.TextGrid.fromFile(grid_file)

        k = 0
        for i in range(len(tg.tiers)):
            for j in range(len(tg.tiers[i])):
                if "YH" not in tg.tiers[i][j].mark:
                    negative = 'none'
                    if len(tg.tiers[i][j].mark.strip())!=0:
                        negative = tg.tiers[i][j].mark
                    xmin = tg.tiers[i][j].minTime
                    xmax = tg.tiers[i][j].maxTime
                    
                    data_negative = data[int(xmin*samplerate):int(xmax*samplerate)]
                    if data_negative.shape[0]/samplerate >= 0.5:
                        nb_clips = int(data_negative.shape[0]/(samplerate*0.5))

                        start = 0
                        end = int(0.5*samplerate)
                        for clip in range(nb_clips):
                            data_clipped = data_negative[start:end]
                            wavfile.write(negative_folder+"clip_"+str(clip+1)+"_"+negative+"_"+str(k)+"_"+sound_file, samplerate, data_clipped)
                            start=end
                            end = end + int(0.5*samplerate)
                        k=k+1

else:
    print("Folder: "+negative_folder+" exists already.")

Folder: /Users/yasminebenhamadi/YellowHammer/Training_set_YH_songs/negative/ exists already.


### Negative bioatic 

In [147]:
def clips_500(folder, outfolder, clipsize=0.5, maxclips=-1):
    files = os.listdir(folder)
    files = [f for f in files if ".wav" in f]
    files = [f for f in files if "speaker" not in f]

    if not os.path.exists(outfolder):
        os.makedirs(outfolder)
    for sound_file in files:
        samplerate, data = wavfile.read(folder+sound_file)

        if data.shape[0]/samplerate >= clipsize:
            nb_clips = int(data.shape[0]/(samplerate*clipsize))
            if maxclips > 0:
                nb_clips = np.min([nb_clips,maxclips])

            start = 0
            end = int(clipsize*samplerate)
            for clip in range(nb_clips):
                data_clipped = data[start:end]
                wavfile.write(outfolder+"clip_"+str(clip+1)+"_"+sound_file, samplerate, data_clipped)
                start=end
                end = end + int(clipsize*samplerate)

In [148]:
bioneg_folder = "/Users/yasminebenhamadi/YellowHammer/Training_set_YH_songs 2/Negative_samples_241016/"
negative_folder = "/Users/yasminebenhamadi/YellowHammer/Training_set_YH_songs/negative_halfhalf/"

clips_500(bioneg_folder, negative_folder)

/var/folders/zv/_mc0pzhx1n713shby01fv4qc0000gq/T/ipykernel_6189/1810236690.py:9: WavFileWarning: Chunk (non-data) not understood, skipping it.
  samplerate, data = wavfile.read(folder+sound_file)
